# GitHub → Colab 更新取得テスト

GitHub `main` の raw URL から毎回 `colab_connection_test.py` を取得し、取得したファイルを実行します。Python標準ライブラリだけを使い、GPU・モデル・追加依存は必要ありません。

実行結果には revision marker、Python、OS、Colab判定、固定計算の検証結果が表示されます。GitHub側の更新後にこのセルを再実行すると、取得したスクリプトの revision marker で更新を確認できます。取得または実行に失敗した場合はセルを失敗として終了します。

In [ ]:
import subprocess
import sys
import tempfile
import urllib.error
import urllib.request
from pathlib import Path

RAW_URL = (
    "https://raw.githubusercontent.com/jialanfudao-stack/git-practice/"
    "main/colab_connection_test.py"
)
print(f"GitHub raw取得元: {RAW_URL}")

try:
    with tempfile.TemporaryDirectory(prefix="colab_connection_test_") as temp_dir:
        script_path = Path(temp_dir) / "colab_connection_test.py"
        try:
            with urllib.request.urlopen(RAW_URL, timeout=30) as response:
                script_path.write_bytes(response.read())
        except (urllib.error.URLError, TimeoutError, OSError) as exc:
            raise RuntimeError(f"GitHub rawの取得に失敗しました: {exc}") from exc

        if not script_path.is_file() or script_path.stat().st_size == 0:
            raise RuntimeError("GitHubから空のスクリプトを取得しました")

        completed = subprocess.run(
            [sys.executable, str(script_path)],
            check=False,
            capture_output=True,
            text=True,
        )
        if completed.stdout:
            print(completed.stdout, end="")
        if completed.stderr:
            print(completed.stderr, end="", file=sys.stderr)
        if completed.returncode != 0:
            raise RuntimeError(
                f"GitHubから取得したスクリプトが失敗しました (exit={completed.returncode})"
            )
except Exception as exc:
    raise RuntimeError(f"Colab接続確認に失敗しました: {exc}") from exc

print("GitHub mainから取得したスクリプトの実行に成功しました。")